In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.transforms import trainval_transforms, revert_normalization, revert_standardization
from src.dataset import ImageDataset
import torch

annot_path = Path("../data/preprocessed/trainval/annotations.csv")
img_dir = Path("../data/preprocessed/trainval/images")

# test dataset without transforms
dataset = ImageDataset(annot_path, img_dir, transform=trainval_transforms)

In [3]:
from src.model import Model
from torch.utils.data import DataLoader

trainval_dl = DataLoader(dataset, 8, True)
X_batch, y_batch = next(iter(trainval_dl))
model = Model()

model.eval()
with torch.no_grad():
    preds_batch = model(X_batch)

In [4]:
from src.postprocessing import postprocess_preds

postprocessed_preds = postprocess_preds(preds_batch[0])

In [5]:
postprocessed_preds

{'person': [(tensor(0.0002),
   tensor(193.2696),
   tensor(128.4607),
   tensor(190.0638),
   tensor(128.5069)),
  (tensor(4.7868e-05),
   tensor(163.4268),
   tensor(-0.7900),
   tensor(157.9400),
   tensor(-0.4971))],
 'dog': [(tensor(0.0003),
   tensor(65.7098),
   tensor(32.7767),
   tensor(63.1383),
   tensor(30.2411)),
  (tensor(0.0002),
   tensor(129.5240),
   tensor(160.0980),
   tensor(126.6481),
   tensor(159.6342))],
 'pottedplant': [(tensor(0.0001),
   tensor(128.6956),
   tensor(34.8187),
   tensor(125.8230),
   tensor(30.6076)),
  (tensor(9.4743e-05),
   tensor(31.4435),
   tensor(62.9039),
   tensor(31.8179),
   tensor(65.1113))],
 'boat': [(tensor(7.9235e-05),
   tensor(158.6334),
   tensor(33.3617),
   tensor(160.5542),
   tensor(29.8307))],
 'chair': [(tensor(0.0003),
   tensor(192.5184),
   tensor(65.2358),
   tensor(191.0350),
   tensor(63.4994)),
  (tensor(8.7488e-05),
   tensor(161.5906),
   tensor(66.1120),
   tensor(158.1424),
   tensor(61.5526)),
  (tensor(7.9

In [6]:
from src.utilities import get_ground_truth_objects_by_class

ground_truth_objects_by_class = get_ground_truth_objects_by_class(y_batch[0])

In [14]:
len(ground_truth_objects_by_class["person"])

4

In [8]:
tp_fp_by_class = {
    "aeroplane": [],
    "bicycle": [],
    "bird": [],
    "boat": [],
    "bottle": [],
    "bus": [],
    "car": [],
    "cat": [],
    "chair": [],
    "cow": [],
    "diningtable": [],
    "dog": [],
    "horse": [],
    "motorbike": [],
    "person": [],
    "pottedplant": [],
    "sheep": [],
    "sofa": [],
    "train": [],
    "tvmonitor": [],
}

In [9]:
from src.evaluation import find_tp_and_fp

find_tp_and_fp(postprocessed_preds, ground_truth_objects_by_class, 
               tp_fp_by_class)

{'aeroplane': [], 'bicycle': [], 'bird': [], 'boat': [], 'bottle': [], 'bus': [], 'car': [], 'cat': [], 'chair': [], 'cow': [], 'diningtable': [], 'dog': [], 'horse': [], 'motorbike': [], 'person': [], 'pottedplant': [], 'sheep': [], 'sofa': [], 'train': [], 'tvmonitor': []} 

{'person': [(tensor(95.), tensor(0.5000), tensor(121.), tensor(43.5000)), (tensor(46.5000), tensor(0.5000), tensor(113.5000), tensor(163.5000)), (tensor(51.), tensor(4.5000), tensor(115.), tensor(223.5000)), (tensor(67.), tensor(1.), tensor(215.), tensor(223.))]} 

{'aeroplane': [], 'bicycle': [], 'bird': [], 'boat': [], 'bottle': [], 'bus': [], 'car': [], 'cat': [], 'chair': [], 'cow': [], 'diningtable': [], 'dog': [], 'horse': [], 'motorbike': [], 'person': [(tensor(0.0002), True)], 'pottedplant': [], 'sheep': [], 'sofa': [], 'train': [], 'tvmonitor': []} 

{'person': [tensor([-5, -4, -3, -2]), (tensor(46.5000), tensor(0.5000), tensor(113.5000), tensor(163.5000)), (tensor(51.), tensor(4.5000), tensor(115.), ten

In [10]:
tp_fp_by_class

{'aeroplane': [],
 'bicycle': [],
 'bird': [],
 'boat': [(tensor(7.9235e-05), False)],
 'bottle': [(tensor(2.7190e-05), False)],
 'bus': [(tensor(0.0003), False),
  (tensor(0.0002), False),
  (tensor(0.0001), False)],
 'car': [(tensor(0.0003), False),
  (tensor(0.0002), False),
  (tensor(4.2104e-05), False)],
 'cat': [(tensor(0.0001), False)],
 'chair': [(tensor(0.0003), False),
  (tensor(8.7488e-05), False),
  (tensor(7.9772e-05), False)],
 'cow': [],
 'diningtable': [],
 'dog': [(tensor(0.0003), False), (tensor(0.0002), False)],
 'horse': [],
 'motorbike': [],
 'person': [(tensor(0.0002), True), (tensor(4.7868e-05), False)],
 'pottedplant': [(tensor(0.0001), False), (tensor(9.4743e-05), False)],
 'sheep': [(tensor(8.6544e-05), False), (tensor(1.0622e-05), False)],
 'sofa': [(tensor(0.0002), False)],
 'train': [],
 'tvmonitor': [(tensor(1.3250e-05), False)]}

In [12]:
class_object_totals = {
    "aeroplane": 0,
    "bicycle": 0,
    "bird": 0,
    "boat": 0,
    "bottle": 0,
    "bus": 0,
    "car": 0,
    "cat": 0,
    "chair": 0,
    "cow": 0,
    "diningtable": 0,
    "dog": 0,
    "horse": 0,
    "motorbike": 0,
    "person": 0,
    "pottedplant": 0,
    "sheep": 0,
    "sofa": 0,
    "train": 0,
    "tvmonitor": 0,
}

In [13]:
from src.evaluation import count_objects_in_each_class

count_objects_in_each_class(ground_truth_objects_by_class, class_object_totals)

class_object_totals

{'aeroplane': 0,
 'bicycle': 0,
 'bird': 0,
 'boat': 0,
 'bottle': 0,
 'bus': 0,
 'car': 0,
 'cat': 0,
 'chair': 0,
 'cow': 0,
 'diningtable': 0,
 'dog': 0,
 'horse': 0,
 'motorbike': 0,
 'person': 4,
 'pottedplant': 0,
 'sheep': 0,
 'sofa': 0,
 'train': 0,
 'tvmonitor': 0}

In [15]:
ground_truth_objects_by_class = get_ground_truth_objects_by_class(y_batch[1])
ground_truths_objects_by_class

NameError: name 'ground_truths_objects_by_class' is not defined